# Quick Start: Running TimesFM 2.5 on BOOM benchmark

This notebook is adapted from the [GiftEval repository](https://github.com/SalesforceAIResearch/gift-eval/tree/main/notebooks) and shows how to run [TimesFM 2.5](https://huggingface.co/google/timesfm-2.5-200m-pytorch) on the BOOM benchmark.

TimesFM 2.5 is a 200M-parameter decoder-only time series foundation model from Google Research. Key features:
- Decoder-only architecture with continuous quantile head
- Built-in input normalization, flip invariance, and positive inference
- Max context length up to 16,384 (this notebook uses 2,048)

`context_length` is set to 2048 for all evaluations in this notebook.

Make sure you download the BOOM benchmark and set the `BOOM` environment variable correctly before running this notebook.

We will use the `Dataset` class from GiftEval to load the data and run the model.

Download BOOM datasets. Calling `download_boom_benchmark` also sets the `BOOM` environment variable with the correct path, which is needed for running the evals below.

In [ ]:
import os
import json
from dotenv import load_dotenv
from dataset_utils import download_boom_benchmark

boom_path = "ChangeMe"
download_boom_benchmark(boom_path)
load_dotenv()

dataset_properties_map = json.load(open("./boom/boom_properties.json"))
all_datasets = list(dataset_properties_map.keys())
print(len(all_datasets))

In [ ]:
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)

# Instantiate the metrics
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]),
]

## TimesFM 2.5 Predictor

We wrap `timesfm.TimesFM_2p5_200M_torch` in a gluonts-compatible predictor. The pipeline is compiled once per `prediction_length` via `ForecastConfig`. `max_context=2048`.

In [ ]:
from typing import List

import numpy as np
import torch
import timesfm
from gluonts.itertools import batcher
from gluonts.model import Forecast
from gluonts.model.forecast import QuantileForecast
from tqdm.auto import tqdm

torch.set_float32_matmul_precision("high")

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
CONTEXT_LENGTH = 2048


class TimesFm25Predictor:
    def __init__(
        self,
        prediction_length: int,
        quantile_levels: List[float] = QUANTILE_LEVELS,
        max_context: int = CONTEXT_LENGTH,
    ):
        self.prediction_length = prediction_length
        self.quantile_levels = quantile_levels
        self.model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
            "google/timesfm-2.5-200m-pytorch",
            torch_compile=True,
        )
        self.model.compile(
            timesfm.ForecastConfig(
                max_context=max_context,
                max_horizon=prediction_length,
                normalize_inputs=True,
                use_continuous_quantile_head=True,
                force_flip_invariance=True,
                infer_is_positive=True,
                fix_quantile_crossing=True,
            )
        )

    def predict(self, test_data_input, batch_size: int = 256) -> List[Forecast]:
        forecasts = []
        while True:
            try:
                for batch in tqdm(batcher(test_data_input, batch_size=batch_size)):
                    context = [np.array(entry["target"]) for entry in batch]
                    _, quantile_forecast = self.model.forecast(
                        horizon=self.prediction_length,
                        inputs=context,
                    )
                    for i, ts in enumerate(batch):
                        # quantile_forecast[i] has shape (horizon, 1 + n_quantiles)
                        # column 0 is the median/point, columns 1: are the quantiles
                        q_arr = quantile_forecast[i][:, 1:]
                        forecasts.append(
                            QuantileForecast(
                                forecast_arrays=q_arr.T,
                                forecast_keys=list(map(str, self.quantile_levels)),
                                start_date=ts["start"] + len(ts["target"]),
                            )
                        )
                break
            except torch.cuda.OutOfMemoryError:
                print(f"OOM at batch_size {batch_size}, halving")
                batch_size //= 2
                forecasts = []
        return forecasts

## Evaluation

We iterate over all BOOM datasets and terms, run the predictor, and record metrics to `results/{model_name}/all_results.csv`. Multivariate datasets are flattened to univariate (TimesFM 2.5 is univariate only).

In [ ]:
import csv
import logging
import os

from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality
from gift_eval.data import Dataset


class _Filter(logging.Filter):
    def filter(self, record):
        return "The mean prediction is not stored in the forecast data" not in record.getMessage()
logging.getLogger("gluonts.model.forecast").addFilter(_Filter())

model_name = "timesfm_2_5_200m"
output_dir = f"ChangeMe/{model_name}"
os.makedirs(output_dir, exist_ok=True)
csv_file_path = os.path.join(output_dir, "all_results.csv")

with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(
        [
            "dataset",
            "model",
            "eval_metrics/MSE[mean]",
            "eval_metrics/MSE[0.5]",
            "eval_metrics/MAE[0.5]",
            "eval_metrics/MASE[0.5]",
            "eval_metrics/MAPE[0.5]",
            "eval_metrics/sMAPE[0.5]",
            "eval_metrics/MSIS",
            "eval_metrics/RMSE[mean]",
            "eval_metrics/NRMSE[mean]",
            "eval_metrics/ND[0.5]",
            "eval_metrics/mean_weighted_sum_quantile_loss",
            "domain",
            "num_variates",
            "dataset_size",
        ]
    )

for ds_num, ds_name in enumerate(all_datasets):
    print(f"Processing dataset: {ds_name} ({ds_num + 1} of {len(all_datasets)})")
    dataset_term = dataset_properties_map[ds_name]["term"]
    for term in ["short", "medium", "long"]:
        if (term == "medium" or term == "long") and dataset_term == "short":
            continue

        ds_freq = dataset_properties_map[ds_name]["frequency"]
        ds_config = f"{ds_name}/{ds_freq}/{term}"

        # TimesFM 2.5 is univariate only; flatten multivariate datasets
        to_univariate = (
            Dataset(name=ds_name, term=term, to_univariate=False, storage_env_var="BOOM").target_dim != 1
        )
        dataset = Dataset(name=ds_name, term=term, to_univariate=to_univariate, storage_env_var="BOOM")

        season_length = get_seasonality(dataset.freq)
        dataset_size = len(dataset.test_data)
        print(f"Dataset size: {dataset_size}")

        predictor = TimesFm25Predictor(
            prediction_length=dataset.prediction_length,
            max_context=CONTEXT_LENGTH,
        )

        res = evaluate_model(
            predictor,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=256,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=season_length,
        )

        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(
                [
                    ds_config,
                    model_name,
                    res["MSE[mean]"].iloc[0],
                    res["MSE[0.5]"].iloc[0],
                    res["MAE[0.5]"].iloc[0],
                    res["MASE[0.5]"].iloc[0],
                    res["MAPE[0.5]"].iloc[0],
                    res["sMAPE[0.5]"].iloc[0],
                    res["MSIS"].iloc[0],
                    res["RMSE[mean]"].iloc[0],
                    res["NRMSE[mean]"].iloc[0],
                    res["ND[0.5]"].iloc[0],
                    res["mean_weighted_sum_quantile_loss"].iloc[0],
                    dataset_properties_map[ds_name]["domain"],
                    dataset_properties_map[ds_name]["num_variates"],
                    dataset_size,
                ]
            )
        print(f"Results for {ds_name} have been written to {csv_file_path}")

## Results

Running the above cell will generate `all_results.csv` under `results/timesfm_2_5_200m/`.

In [ ]:
import pandas as pd
df = pd.read_csv(output_dir + "/all_results.csv")
df